In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
ω_a = 3.85*(rand(1)[1]/5+0.9)*2*pi
ω_b = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.9
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=2/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
ϵ_0_list = [0.8]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 2200/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
for ϵ_0 in ϵ_0_list

    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
ϵ_0_list = [0.802]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 2200/κ_b
t_min = 0.0
t_min_save = t_min
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
dict = Final_all[0.802]

# Phase-space resolution
nx = 200
ny = 200

# Collect all points once to define bounds
all_points = reduce(vcat, values(dict))
x_all = real.(all_points)
y_all = imag.(all_points)

x_edges = range(minimum(x_all), maximum(x_all); length=nx+1)
y_edges = range(minimum(y_all), maximum(y_all); length=ny+1)

In [ ]:
using StatsBase

function shannon_entropy(weights)
    p = weights ./ sum(weights)
    p = p[p .> 0]                 # avoid log(0)
    return -sum(p .* log.(p))
end
group_size = 1000
trajectories = collect(values(dict))
ntraj = length(trajectories)

# number of chunks (use minimum length to stay safe)
nchunks = minimum(length(traj) ÷ group_size for traj in trajectories)

entropy_per_chunk = zeros(Float64, nchunks, ntraj)

for (j, traj) in enumerate(trajectories)
    for k in 1:nchunks
        i0 = (k-1)*group_size + 1
        i1 = k*group_size
        chunk = @view traj[i0:i1]

        h = fit(
            Histogram,
            (real.(chunk), imag.(chunk)),
            (x_edges, y_edges)
        )

        entropy_per_chunk[k, j] = shannon_entropy(h.weights)
    end
end

nbins_entropy = 500

Hmin = minimum(entropy_per_chunk)
Hmax = maximum(entropy_per_chunk)

entropy_bins = range(Hmin, Hmax; length=nbins_entropy+1)
entropy_centers = (entropy_bins[1:end-1] .+ entropy_bins[2:end]) ./ 2

entropy_hist_matrix = zeros(Float64, nbins_entropy, nchunks)

for k in 1:nchunks
    h = fit(Histogram, entropy_per_chunk[k, :], entropy_bins)
    entropy_hist_matrix[:, k] .= h.weights
end

chunk_indices = (0:nchunks-1) .* group_size .+ 1   # index of first point in chunk
chunk_time_converted = chunk_indices*dt_save*κ_b

In [ ]:

heatmap(
    chunk_time_converted,
    entropy_centers,
    entropy_hist_matrix;
    cmap = :viridis,
    xlabel = L"\kappa_bt",
    ylabel = "Shannon entropy",
    colorbar = true,
    #ylims=(5,7)
)

In [ ]:
using EasyFit
H_thresh_chaos = 5.0   # example threshold (choose physically meaningful value)
proportion_chaos = [count(>=(H_thresh_chaos), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

H_thresh_reg = 1.0
proportion_reg = [count(<=(H_thresh_reg), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

fit_exp_c = fitexp(chunk_time_converted[1:end-5],proportion_chaos)
fit_exp_r = fitexp(chunk_time_converted[1:end-5],proportion_reg)

A_chaos,B_chaos,C_chaos = fit_exp_c.a,fit_exp_c.b,fit_exp_c.c
A_reg,B_reg,C_reg = fit_exp_r.a,fit_exp_r.b,fit_exp_r.c

time_lis = 0:1:maximum(chunk_time_converted)
prop_chaos_lis =@. A_chaos*exp(-time_lis/B_chaos)+C_chaos
prop_reg_lis = @. A_reg*exp(-time_lis/B_reg)+C_reg;

In [ ]:
plot(
    chunk_time_converted[1:end-5],
    proportion_chaos,
    lw = 2,
    label = "Chaotic traj."
)
plot!(
    time_lis,
    prop_chaos_lis,
    ls =:dash,
    label=""
)
plot!(
    chunk_time_converted[1:end-5],
    proportion_reg,
    lw = 2,
    label = "Regular traj."
)
plot!(
    time_lis,
    prop_reg_lis,
    ls =:dash,
    label=""
)
plot!(
    xlabel = L"\kappa_bt",
    ylabel = "Fraction of trajectories",
#    ylims = (0.0, 1)
)

#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/chaos/transient_chaos_23_03_2026.pdf")